In [1]:
import torch
import os
import glob
from PIL import Image
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
from tqdm import tqdm

# --- IMPOSTAZIONE DEL DEVICE (GPU o CPU) ---
# Questa cella definisce il device una volta per tutte.
# Tutte le altre parti del codice useranno questa variabile.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device in uso: {device}")


print(f"Current working directory: {os.getcwd()}")

COLAB = False;
if COLAB:
    # Montiamo Google Drive
    from google.colab import drive
    drive.mount('/content/drive')

    # Definiamo il percorso principale
    DATASET_PATH = "/content/drive/MyDrive/Uni/Magistrale/colab/1/ML/Project/training/train"
else:
    DATASET_PATH = "../../dataset/training/train"

# Se si vuole eseguire tutto solo nel main
EXECUTE_MAIN = False


Device in uso: cuda
Current working directory: c:\Users\leona\OneDrive\Documenti\progettoML\notebooks\first_model


# SegmentationDataset

In [2]:
# --- DEFINIZIONE DELLA CLASSE DATASET ---
import os
from PIL import Image
import numpy as np
import torch
from torch.utils.data import Dataset

class SegmentationDataset(Dataset):
    """
    Versione definitiva e semplificata del Dataset.
    Legge le immagini e le maschere che contengono già gli ID di classe.
    """
    def __init__(self, sample_dirs, transform=None):
        self.sample_dirs = sample_dirs
        self.transform = transform

    def __len__(self):
        return len(self.sample_dirs)

    def __getitem__(self, idx):
        sample_dir = self.sample_dirs[idx]

        img_path = os.path.join(sample_dir, 'rgb.jpg')
        mask_path = os.path.join(sample_dir, 'labels.png')

        # Carichiamo l'immagine RGB e la maschera
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path) # È già in scala di grigi con indici 0-8

        # Convertiamo in array NumPy
        # Normalizziamo l'immagine a [0, 1] per la stabilità del training
        image_np = np.array(image, dtype=np.float32) / 255.0
        # Le maschere contengono ID, quindi usiamo un tipo intero
        mask_np = np.array(mask, dtype=np.int64)

        if self.transform:
            # Qui applicheremo la data augmentation nelle fasi successive
            pass

        # Convertiamo in tensori PyTorch
        # Immagine -> [C, H, W]
        image_tensor = torch.from_numpy(image_np).permute(2, 0, 1)
        # Maschera -> [H, W] (la loss CrossEntropy vuole questo formato)
        mask_tensor = torch.from_numpy(mask_np)

        return image_tensor, mask_tensor

In [3]:
import glob
from torch.utils.data import random_split
from typing import Tuple, List

def create_splits(data_dir: str, val_percent: float = 0.2, seed: int = 42) -> Tuple[List[str], List[str]]:
    """
    Trova tutti i campioni e li divide in liste di percorsi per training e validation.
    """
    # 1. Trova tutte le directory dei campioni validi
    all_sample_dirs = sorted([d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d) and not os.path.basename(d).startswith('__')])
    print(f"Trovati {len(all_sample_dirs)} campioni totali validi.")

    # 2. Calcola le dimensioni degli split
    n_val = int(len(all_sample_dirs) * val_percent)
    n_train = len(all_sample_dirs) - n_val
    
    # 3. Esegui lo split
    generator = torch.Generator().manual_seed(seed)
    train_dirs, val_dirs = random_split(all_sample_dirs, [n_train, n_val], generator=generator)
    
    print(f"Split creato: {len(train_dirs)} campioni per training, {len(val_dirs)} per validazione.")
    
    # Restituisce le liste di percorsi
    return list(train_dirs), list(val_dirs)


if not EXECUTE_MAIN:
  train_dirs, val_dirs = create_splits(DATASET_PATH)
  train_dataset = SegmentationDataset(sample_dirs=train_dirs)
  val_dataset = SegmentationDataset(sample_dirs=val_dirs)
  print(f"Oggetto train_dataset creato con {len(train_dataset)} campioni.")

Trovati 931 campioni totali validi.
Split creato: 745 campioni per training, 186 per validazione.
Oggetto train_dataset creato con 745 campioni.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- MATTONI DI COSTRUZIONE DELLA U-NET ---

class DoubleConv(nn.Module):
    """(Convolution => BatchNorm => ReLU) * 2"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class Down(nn.Module):
    """Downscaling con MaxPool2d seguito da DoubleConv"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)

class Up(nn.Module):
    """Upscaling (con ConvTranspose2d) seguito da DoubleConv"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # Usiamo ConvTranspose2d per l'upsampling, che è un layer addestrabile
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        # x1 è il tensore dal percorso di upscaling
        # x2 è la skip connection dal percorso di encoding
        x1 = self.up(x1)
        # Concatena la skip connection con il tensore upscalato
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class OutConv(nn.Module):
    """Convoluzione 1x1 finale per mappare ai canali delle classi"""
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)



# --- ARCHITETTURA U-NET COMPLETA ---
class UNet(nn.Module):
    def __init__(self, n_channels, n_classes):
        super(UNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes

        # Encoder
        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024)
        
        # Decoder
        self.up1 = Up(1024, 512)
        self.up2 = Up(512, 256)
        self.up3 = Up(256, 128)
        self.up4 = Up(128, 64)
        self.outc = OutConv(64, n_classes)

    def forward(self, x):
        # Percorso dell'encoder
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        
        # Percorso del decoder con le skip connections
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        
        # Output finale (logits)
        logits = self.outc(x)
        return logits
    
# --- ARCHITETTURA U-NET LITE (PIÙ LEGGERA) ---
class UNetLite(nn.Module):
    def __init__(self, n_channels, n_classes):
        super(UNetLite, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes

        # Encoder - Canali ridotti
        self.inc = DoubleConv(n_channels, 16)
        self.down1 = Down(16, 32)
        self.down2 = Down(32, 64)
        self.down3 = Down(64, 128)
        # Non andiamo più in profondità per risparmiare memoria
        
        # Decoder - Canali ridotti
        self.up1 = Up(128, 64)
        self.up2 = Up(64, 32)
        self.up3 = Up(32, 16)
        self.outc = OutConv(16, n_classes)

    def forward(self, x):
        # Percorso dell'encoder
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        # Non c'è x5
        
        # Percorso del decoder con le skip connections
        x = self.up1(x4, x3)
        x = self.up2(x, x2)
        x = self.up3(x, x1)
        
        # Output finale (logits)
        logits = self.outc(x)
        return logits
    
# --- ARCHITETTURA U-NET MICRO (ANCORA PIÙ LEGGERA) ---
class UNetMicro(nn.Module):
    def __init__(self, n_channels, n_classes):
        super(UNetMicro, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes

        # Canali di base molto più piccoli
        self.inc = DoubleConv(n_channels, 16)
        self.down1 = Down(16, 32)
        self.down2 = Down(32, 64)
        # Rimuoviamo anche il terzo stadio di downsampling per ora
        # self.down3 = Down(64, 128) 
        
        # Il punto più profondo ora ha 64 canali
        
        # Decoder
        # self.up1 = Up(128, 64)
        self.up2 = Up(64, 32)
        self.up3 = Up(32, 16)
        self.outc = OutConv(16, n_classes)

    def forward(self, x):
        # Percorso dell'encoder
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        # x4 = self.down3(x3)
        
        # Percorso del decoder
        # x = self.up1(x4, x3)
        x = self.up2(x3, x2) # Parte da x3, la skip è x2
        x = self.up3(x, x1)  # Continua con x1
        
        logits = self.outc(x)
        return logits


In [5]:
from torchvision import transforms
from tqdm import tqdm

def calculate_mean_std(dataset):
    # Usiamo un DataLoader solo per questa funzione per efficienza
    loader = DataLoader(dataset, batch_size=16, shuffle=False, num_workers=0)
    
    mean = torch.zeros(3)
    std = torch.zeros(3)
    total_images_count = 0
    
    for images, _ in tqdm(loader, desc="Calcolo Mean/Std"):
        batch_samples = images.size(0)
        images = images.view(batch_samples, images.size(1), -1)
        mean += images.mean(2).sum(0)
        std += images.std(2).sum(0)
        total_images_count += batch_samples
        
    mean /= total_images_count
    std /= total_images_count
    
    return mean, std

# Ora 'train_dataset' è definito e possiamo usarlo
print("Calcolo di Mean e Std sul training set...")
mean, std = calculate_mean_std(train_dataset)

print(f"\nMean calcolato: {mean}")
print(f"Std calcolato: {std}")

Calcolo di Mean e Std sul training set...


Calcolo Mean/Std: 100%|██████████| 47/47 [00:15<00:00,  3.12it/s]


Mean calcolato: tensor([0.4278, 0.4464, 0.4231])
Std calcolato: tensor([0.2543, 0.2535, 0.2567])


In [6]:
from torchvision import transforms

# Definiamo le trasformazioni. Per ora solo la normalizzazione.
# La applicheremo direttamente nel ciclo di training per semplicità.
normalize_transform = transforms.Normalize(mean=mean, std=std)

# Creiamo i DataLoader finali
BATCH_SIZE = 8 # Iniziamo con un valore sicuro
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Imposta il device (GPU se disponibile)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device in uso: {device}")

# Istanzia il modello
# n_channels = 3 per immagini RGB
# n_classes = 9 per le nostre classi da 0 a 8
model = UNet(n_channels=3, n_classes=9).to(device)

# Creiamo un tensore di input fittizio (dummy) con le dimensioni di un'immagine
# Usiamo un batch_size di 1 per misurare il consumo di memoria per un singolo campione in fase di test
dummy_input = torch.randn(1, 3, 544, 1024).to(device)

# --- PROFILING MEMORIA E TEMPO DI INFERENZA ---
if device.type == 'cuda':
    # Sincronizza per una misurazione accurata
    torch.cuda.synchronize()
    # Pulisci la cache della memoria
    torch.cuda.empty_cache()
    # Misura la memoria prima dell'inferenza
    mem_before = torch.cuda.memory_allocated(device)

# Eseguiamo il forward pass per il profiling
with torch.no_grad():
    output = model(dummy_input)

if device.type == 'cuda':
    torch.cuda.synchronize()
    # Misura la memoria dopo l'inferenza
    mem_after = torch.cuda.memory_allocated(device)
    
    memory_usage_gb = (mem_after - mem_before) / (1024 ** 3)
    print(f"Forma dell'output del modello: {output.shape}")
    print(f"Memoria GPU usata per l'inferenza (test, batch_size=1): {memory_usage_gb:.4f} GB")

    # Verifica rispetto al vincolo
    if memory_usage_gb < 4.0:
        print("Il modello rispetta il vincolo di memoria per il test (< 4 GB).")
    else:
        print("ATTENZIONE: Il modello supera il vincolo di memoria per il test (> 4 GB).")

Device in uso: cuda
Forma dell'output del modello: torch.Size([1, 9, 544, 1024])
Memoria GPU usata per l'inferenza (test, batch_size=1): 0.0187 GB
Il modello rispetta il vincolo di memoria per il test (< 4 GB).


In [7]:
import torch.optim as optim
from torchvision import transforms

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF

# --- SETUP GLOBALE ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Valori calcolati nella cella precedente
DATASET_MEAN = [0.4278, 0.4464, 0.4231] 
DATASET_STD = [0.2543, 0.2535, 0.2567]

# Riduciamo drasticamente sia il batch size che la risoluzione per la nostra baseline
BATCH_SIZE = 1
OUTPUT_SIZE = (1024, 544)


# --- TRASFORMAZIONI E DATASET ---
# Ridefiniamo la classe Dataset per includere le trasformazioni
class SegmentationDataset(Dataset):
    def __init__(self, sample_dirs, output_size, mean, std, is_train=True):
        self.sample_dirs = sample_dirs
        self.output_size = output_size
        # Definiamo le trasformazioni direttamente qui
        self.normalize = transforms.Normalize(mean=mean, std=std)
        self.is_train = is_train

    def __len__(self):
        return len(self.sample_dirs)

    def __getitem__(self, idx):
        sample_dir = self.sample_dirs[idx]
        img_path = os.path.join(sample_dir, 'rgb.jpg')
        mask_path = os.path.join(sample_dir, 'labels.png')
        
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)

        # Ridimensionamento
        image = TF.resize(image, self.output_size, interpolation=TF.InterpolationMode.BILINEAR)
        mask = TF.resize(mask, self.output_size, interpolation=TF.InterpolationMode.NEAREST)
        
        # --- DATA AUGMENTATION (solo per il training set) ---
        if self.is_train:
            # Esempio: flip orizzontale con probabilità 50%
            if torch.rand(1) < 0.5:
                image = TF.hflip(image)
                mask = TF.hflip(mask)

        # Conversione a tensore e normalizzazione
        image = TF.to_tensor(image)
        image = self.normalize(image)
        mask = torch.from_numpy(np.array(mask, dtype=np.int64))

        return image, mask

# Creiamo le istanze del dataset
train_dataset = SegmentationDataset(train_dirs, OUTPUT_SIZE, DATASET_MEAN, DATASET_STD, is_train=True)
val_dataset = SegmentationDataset(val_dirs, OUTPUT_SIZE, DATASET_MEAN, DATASET_STD, is_train=False)

# Creiamo i DataLoader con pin_memory=True
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)


# --- MODELLO, LOSS, OTTIMIZZATORE ---
model = UNet(n_channels=3, n_classes=9).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3) # Aumentiamo leggermente il learning rate
criterion = nn.CrossEntropyLoss(ignore_index=0) # Ignoriamo la classe background


# --- FUNZIONE DI VALUTAZIONE ---
def evaluate(model, loader, device, num_classes):
    model.eval()
    all_ious = []
    
    with torch.no_grad():
        for images, masks in tqdm(loader, desc="Validazione"):
            images = images.to(device)
            masks_np = masks.numpy() # Ground truth per il calcolo IoU
            
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            
            # Calcola IoU per ogni classe in ogni immagine del batch
            for i in range(images.size(0)):
                iou_scores = compute_all_iou(masks_np[i], preds[i], num_labels=num_classes-1)
                all_ious.append(iou_scores)
    
    # Calcola la media su tutte le immagini e tutte le classi, ignorando i NaN
    mean_iou = np.nanmean(all_ious)
    return mean_iou

def compute_all_iou(mask_true, mask_pred, num_labels):
    iou_scores = np.zeros(num_labels)
    for label in range(1, num_labels + 1): # Salta il background (classe 0)
        intersection = np.sum((mask_true == label) & (mask_pred == label))
        union = np.sum((mask_true == label) | (mask_pred == label))
        if union == 0:
            iou_scores[label-1] = np.nan
        else:
            iou_scores[label-1] = intersection / union
    return iou_scores


# --- CICLO DI TRAINING E VALIDAZIONE ---
num_epochs = 10 # Aumentiamo le epoche per un training significativo
best_iou = -1.0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Training]"):
        images = images.to(device)
        masks = masks.to(device, dtype=torch.long)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    epoch_loss = running_loss / len(train_loader)
    
    # Validazione alla fine di ogni epoca
    val_iou = evaluate(model, val_loader, device, num_classes=9)
    
    print(f"Epoch {epoch+1} - Training Loss: {epoch_loss:.4f} - Validation mIoU: {val_iou:.4f}")
    
    # Salva il modello migliore
    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), 'best_model_baseline.pth')
        print(f"🎉 Nuovo modello migliore salvato con mIoU: {best_iou:.4f}")

Validazione: 100%|██████████| 186/186 [00:32<00:00,  5.73it/s]


Epoch 1 - Training Loss: 1.1746 - Validation mIoU: 0.2813
🎉 Nuovo modello migliore salvato con mIoU: 0.2813


Validazione: 100%|██████████| 186/186 [00:32<00:00,  5.69it/s]


Epoch 2 - Training Loss: 1.0191 - Validation mIoU: 0.3285
🎉 Nuovo modello migliore salvato con mIoU: 0.3285


Validazione: 100%|██████████| 186/186 [00:32<00:00,  5.66it/s]


...

Epoch 8 - Training Loss: 0.7664 - Validation mIoU: 0.3070


Validazione: 100%|██████████| 186/186 [00:33<00:00,  5.63it/s]


Epoch 9 - Training Loss: 0.7518 - Validation mIoU: 0.3085


Validazione: 100%|██████████| 186/186 [00:33<00:00,  5.52it/s]

Epoch 10 - Training Loss: 0.7217 - Validation mIoU: 0.2940
